In [0]:
# pip install --upgrade huggingface-hub

In [0]:
%pip install mlflow xformers sentence-transformers==3.1.0 flash-attn

# https://tridao.me/publications/flash3/flash3.pdf
# https://github.com/Dao-AILab/flash-attention/blob/main/README.md

Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.3/28.3 MB 68.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 90.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.1/249.1 kB 44.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 118.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 29.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 124.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 30.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.6/233.6 kB 46.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 31.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.3/64.3 kB 17.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 643.8/643.8 kB 82.2 MB/s eta 0:00:00
     ━━

In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd
import mlflow
from datasets import load_dataset, load_from_disk
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    SentenceTransformerModelCardData,
)
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers, ParallelMode
from sentence_transformers.evaluation import TripletEvaluator

Unexpected internal error when monkey patching `PreTrainedModel.from_pretrained`: Failed to import transformers.modeling_utils because of the following error (look up to see its traceback):
/databricks/python/lib/python3.10/site-packages/flash_attn_2_cuda.cpython-310-x86_64-linux-gnu.so: undefined symbol: _ZN2at4_ops9_pad_enum4callERKNS_6TensorEN3c108ArrayRefINS5_6SymIntEEElNS5_8optionalIdEE
Unexpected internal error when monkey patching `Trainer.train`: Failed to import transformers.trainer because of the following error (look up to see its traceback):
Failed to import transformers.integrations.integration_utils because of the following error (look up to see its traceback):
Failed to import transformers.modeling_utils because of the following error (look up to see its traceback):
/databricks/python/lib/python3.10/site-packages/flash_attn_2_cuda.cpython-310-x86_64-linux-gnu.so: undefined symbol: _ZN2at4_ops9_pad_enum4callERKNS_6TensorEN3c108ArrayRefINS5_6SymIntEEElNS5_8optionalIdEE


---------------------------------------------------------------------------
ImportError                               Traceback (most recent call last)
File /local_disk0/.ephemeral_nfs/envs/pythonEnv-ec66b5f7-5b8b-4231-81cd-488f9dc7a1ca/lib/python3.10/site-packages/transformers/utils/import_utils.py:1817, in _LazyModule._get_module(self, module_name)
   1816 try:
-> 1817     return importlib.import_module("." + module_name, self.__name__)
   1818 except Exception as e:

File /usr/lib/python3.10/importlib/__init__.py:126, in import_module(name, package)
    125         level += 1
--> 126 return _bootstrap._gcd_import(name[level:], package, level)

File <frozen importlib._bootstrap>:1050, in _gcd_import(name, package, level)

File <frozen importlib._bootstrap>:1027, in _find_and_load(name, import_)

File <frozen importlib._bootstrap>:1006, in _find_and_load_unlocked(name, import_)

File <frozen importlib._bootstrap>:688, in _load_unlocked(spec)

File <frozen importlib._bootstrap_external

In [0]:
# set the experiment id
mlflow.set_experiment(experiment_id="520f9932547e442eaa0a940db3cc168c")   # for shared-dev
# mlflow.set_experiment(experiment_id="90112531520980")   # for exp: finetune_transformers: all remaining models (matching-dev)
# mlflow.set_experiment(experiment_id="724741476774450")   # for exp: finetune_transformers2 : gte-large-en models (matching-dev)
# mlflow.set_experiment(experiment_id="1791332171056036") # for exp: finetune_transformers3 : models (matching-dev)

In [0]:
# dataset1: hard negatives
# dataset2: medium negatives
# dataset3: easy neg
dataset_path = '/Volumes/ctlg_insight_dev_gold/schm_search/mroteb_training_datasets/dataset2'
train_path = dataset_path + '/dataset_train'
eval_path = dataset_path + '/dataset_eval'
test_path = dataset_path + '/dataset_test'
dataset_train = load_from_disk(train_path)
dataset_eval = load_from_disk(eval_path)
dataset_test = load_from_disk(test_path)

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:136)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecutio

In [0]:
import pyarrow as pa

try:
    with pa.input_stream("/Volumes/ctlg_insight_dev_gold/schm_search/mroteb_training_datasets/dataset2/dataset_test/data-00000-of-00001.arrow") as source:
        reader = pa.ipc.open_stream(source)
        table = reader.read_all()
        print(table)
except Exception as e:
    print(f"Validation failed: {e}")

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:447)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperation$1(UsageLogging.scala:573)
	at com.databricks.logging.UsageLogging.executeThunkAndCaptureResultTags$1(UsageLogging.scala:669)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperationWithResultTags$4(UsageLogging.scala:687)
	at com.databricks.logging.UsageLogging.$anonfun$withAttributionContext$1(UsageLogging.scala:426)
	at scala.util.DynamicVariable.withValue(DynamicVariable.scala:62)
	at com.databricks.logging.AttributionContext$.withValue(AttributionContext.scala:216)
	at com.databricks.logging.UsageLogging.withAttributionContext(UsageLogging.scala:424)
	at com.databricks.logging.Usa

In [0]:
# # for dataset1: <model_name>_ft_v<version_number>
# # for dataset2, dataset3: <model_name>_ft_v<version_number>.<dataset_number>
ft_model_name = 'multilingual-e5-large-ft-v1.2'

In [0]:
from huggingface_hub import snapshot_download
from sentence_transformers import (
    SentenceTransformer
)

# model_name = "BAAI/bge-large-en-v1.5"
model_name = "intfloat/multilingual-e5-large"   # change model path name
HF_ENDPOINT = 'https://graingerreadonly.jfrog.io/artifactory/api/huggingfaceml/huggingfaceml-remote'  

model_path = snapshot_download(
    repo_id=model_name,
    etag_timeout=86400,
    endpoint= HF_ENDPOINT
)
model = SentenceTransformer(model_name_or_path=model_path,trust_remote_code=True)

In [0]:
# model_path = "microsoft/mpnet-base"
# model_path = "sentence-transformers/all-mpnet-base-v2"
# model = SentenceTransformer(model_path)
# model = SentenceTransformer(model_path,trust_remote_code=True)
data = ["This is an example sentence", "Each sentence is converted"]
embeddings = model.encode(data)
embeddings

In [0]:
 # num_epocs = 1 for v1 and num_epocs = 3 for v2

In [0]:
with mlflow.start_run(run_name=f"{ft_model_name}") as run:
    # 1. Load a model to finetune with 2. (Optional) model card data
    model = SentenceTransformer(
        # "BAAI/bge-large-en-v1.5",
        model_path, # for shared-dev (downloading from graingerreadonly.jfrog.io)
        trust_remote_code=True,
        model_card_data=SentenceTransformerModelCardData(
            language="en",
            model_name=f"{ft_model_name}"
        )
    )

    # 3. Load a dataset to finetune on
    train_dataset = load_from_disk(train_path)
    eval_dataset = load_from_disk(eval_path)
    test_dataset = load_from_disk(test_path).select(range(1_000))

    # 4. Define a loss function
    loss = MultipleNegativesRankingLoss(model)

    # 5. Specify training arguments
    num_epochs = 1 # v1: num_epocs = 1, v2: num_epocs = 3
    learning_rate = 2e-5
    train_batch_size = 32
    gradient_accumulation_steps = 4

    args = SentenceTransformerTrainingArguments(
        output_dir=f"/{ft_model_name}",
        num_train_epochs=num_epochs,
        per_device_train_batch_size=train_batch_size,
        per_device_eval_batch_size=train_batch_size,
        learning_rate=learning_rate,
        warmup_ratio=0.1,
        fp16=True,
        # fp16=False,
        #fp16:Enables mixed precision training, where computations are performed using 16-bit floating point numbers instead of 32-bit.This reduces memory usage and can speed up training on GPUs with tensor core support (e.g., NVIDIA V100 or A100).Use cautiously as it may introduce numerical instability in some cases.
        bf16=False,
        #bf16:Enables bfloat16 precision training, similar to fp16, but with greater numerical stability. Useful on hardware like TPUs or GPUs supporting bfloat16.
        batch_sampler=BatchSamplers.NO_DUPLICATES,
        evaluation_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        logging_steps=100,
        run_name=f"{ft_model_name}",
        gradient_accumulation_steps=gradient_accumulation_steps,
        dataloader_drop_last=True
    )

    # Log parameters with MLflow
    mlflow.log_param("num_train_epochs", num_epochs)
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_param("train_batch_size", train_batch_size)
    mlflow.log_param("loss_function", "MultipleNegativesRankingLoss")
    mlflow.log_param("gradient_accumulation_steps", gradient_accumulation_steps)

    # 6. Create an evaluator & evaluate the base model
    dev_evaluator = TripletEvaluator(
        anchors=eval_dataset["anchor"],
        positives=eval_dataset["positive"],
        negatives=eval_dataset["negative"],
        name="dev",
    )
    dev_evaluator_results = dev_evaluator(model)

    # Log dev evaluator results
    for metric, value in dev_evaluator_results.items():
        mlflow.log_metric(f"dev_{metric}", value)

    # 7. Create a trainer & train
    trainer = SentenceTransformerTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        loss=loss,
        evaluator=dev_evaluator,
    )

    trainer.train()

    # 8. (Optional) Evaluate the trained model on the test set
    test_evaluator = TripletEvaluator(
        anchors=test_dataset["anchor"],
        positives=test_dataset["positive"],
        negatives=test_dataset["negative"],
        name="test",
    )
    test_evaluator_results = test_evaluator(model)

    # Log test evaluator results
    for metric, value in test_evaluator_results.items():
        mlflow.log_metric(f"test_{metric}", value)

    # 8. Save the trained model as an artifact
    model_path = f"/Volumes/ctlg_insight_dev_gold/schm_search/mroteb_models/{ft_model_name}" # for shared-dev
    # model_path = f"/dbfs/FileStore/test_priyanka/finetuned_models/{ft_model_name}"   # for matching-dev-1
    model.save_pretrained(model_path)

    # Log the model as an artifact
    mlflow.log_artifact(model_path)

    # infer signature
    data = ["This is an example sentence", "Each sentence is converted"]
    # signature = mlflow.models.signature.infer_signature(data, embeddings)
    signature = mlflow.models.infer_signature(
    model_input=data,
    model_output=model.encode(data),
    )

    mlflow.sentence_transformers.log_model(
        model=model,
        artifact_path=f"{ft_model_name}",
        signature=signature,
        input_example=data
    )

# Evaluate ft model

In [0]:
# import mroteb
# import torch
# #import mlflow
# mlflow.autolog(disable=True)
# # Check if GPU is available
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Using device: {device}")
# # device = "cuda" if torch.has_mps else "cpu"
# # print(device)

In [0]:
# task_names = ["MroBrandQueriesDbx","MroPartNumQueriesDbx","MroAttrAlphaNumQueriesDbx", "MroAttrAlphaOnlyQueriesDbx", "MroAttrNumOnlyQueriesDbx","MroGenericQueriesDbx","MroDBXComp","MroDBXPopints","MroDBXInternal","MroDBXLNdesc","MroDBXFTXSTS"]
# # ["MroBrandQueriesDbx","MroPartNumQueriesDbx","MroAttrAlphaNumQueriesDbx", "MroAttrAlphaOnlyQueriesDbx", "MroAttrNumOnlyQueriesDbx","MroGenericQueriesDbx"]
# tasks = mroteb.get_tasks(tasks=task_names)

In [0]:
# model_path = f"/Volumes/ctlg_insight_dev_gold/schm_search/mroteb_models//{ft_model_name}"
# print(model_path)
# model_name = f"{ft_model_name}"
# model = SentenceTransformer(model_path, trust_remote_code = True, device=device)

In [0]:
# eval_folder = "/Workspace/Users/priyanka.bhosale@grainger.com/Semantic_search/eval_results/mroteb"
# output_folder = eval_folder + '/'+ model_name + '/'

In [0]:
# # import torch
# torch.cuda.empty_cache()

In [0]:
# evaluation = mroteb.MTEB(tasks=tasks)
# # res = evaluation.run(model, output_folder=output_folder, overwrite_results=False)
# res = evaluation.run(model, output_folder=output_folder, overwrite_results=False, encode_kwargs ={'batch_size': 32}) # Give smaller batch size for large models to avoid out of memory error
# res[0].scores